In [2]:
import json
from itertools import groupby

from src.config import Config, DiffusionConfig

In [3]:
run_ids = [
    "hybrid_conf_2026-09-04T22_29_24_rep1",
    "hybrid_conf_2026-09-05T11_41_27_rep1",
    "hybrid_conf_2026-09-05T11_45_47_rep1",
    "hybrid_conf_2026-09-05T10_41_43_rep1",
    "hybrid_conf_2026-09-05T10_39_12_rep1",
    "hybrid_conf_2026-09-05T10_34_21_rep1",
    "hybrid_conf_2026-09-05T10_32_53_rep1",
    "hybrid_conf_2026-09-05T10_31_12_rep1",
]

config_dicts = {}
val_result_dicts = {}
for run_id in run_ids:
    config_path = f"results/results_{run_id}.json"
    with open(config_path) as f:
        result_dict = json.load(f)
    config_dicts[run_id] = result_dict["config"]
    val_result_dicts[run_id] = result_dict["result_val"]

In [4]:
def all_equal(iterable):
    g = groupby(iterable)
    return next(g, True) and not next(g, False)


# Make sure configs all match except for the diffusion subconfig cf_anchor_weight

for subconfig in ["vae", "train", "data"]:
    assert all_equal([config_dict[subconfig] for config_dict in config_dicts.values()])

for diffusion_subconfig in DiffusionConfig.model_fields:
    assert all_equal(
        [
            config_dict["diffusion"][diffusion_subconfig]
            for config_dict in config_dicts.values()
            if diffusion_subconfig != "cf_anchor_weight"
        ]
    )

run_id_to_anchor_weight = {
    run_id: config_dict["diffusion"]["cf_anchor_weight"]
    for run_id, config_dict in config_dicts.items()
}
run_id_to_anchor_weight

{'hybrid_conf_2026-09-04T22_29_24_rep1': 1.0,
 'hybrid_conf_2026-09-05T11_41_27_rep1': 0.97,
 'hybrid_conf_2026-09-05T11_45_47_rep1': 0.96,
 'hybrid_conf_2026-09-05T10_41_43_rep1': 0.95,
 'hybrid_conf_2026-09-05T10_39_12_rep1': 0.9,
 'hybrid_conf_2026-09-05T10_34_21_rep1': 0.75,
 'hybrid_conf_2026-09-05T10_32_53_rep1': 0.5,
 'hybrid_conf_2026-09-05T10_31_12_rep1': 0.0}

In [12]:
config_objs = {run_id: Config(**config_dict) for run_id, config_dict in config_dicts.items()}

anchor_weight_to_val_results = {}
for run_id in run_ids:
    anchor_weight = run_id_to_anchor_weight[run_id]
    val_results = val_result_dicts[run_id]
    anchor_weight_to_val_results[anchor_weight] = val_results

# print("w_anchor", " ".join(anchor_weight_to_val_results[1].keys()))
for w, res in anchor_weight_to_val_results.items():
    print(f"{w}\t" + "\t".join([f"{v:.2f}" for v in res.values()]))

1.0	2.41	0.94	0.99	1.00	18.12	8.71	0.99	1.00	21.49	13.78	1.62	0.65	1.74
0.97	2.10	1.35	0.99	1.00	16.47	13.16	0.99	1.00	20.83	18.86	1.36	0.74	1.49
0.96	3.72	4.29	1.00	1.00	23.92	28.47	1.00	1.00	27.86	31.78	1.45	1.47	2.28
0.95	2.79	2.68	1.00	1.00	21.04	23.41	1.00	1.00	26.43	28.60	1.27	0.93	1.49
0.9	4.09	5.82	1.00	1.00	24.31	32.98	1.00	1.00	26.71	34.30	1.38	1.37	2.04
0.75	4.00	5.39	1.00	1.00	24.90	32.68	1.00	1.00	28.39	34.06	1.34	1.38	1.99
0.5	7.22	8.99	1.00	1.00	30.29	34.72	1.00	1.00	31.68	35.01	1.68	1.81	2.84
0.0	5.84	10.29	1.00	1.00	26.00	34.94	1.00	1.00	27.41	35.06	1.60	2.30	3.37
